In [2]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit
from astropy.modeling import models
from scipy.signal import convolve
from scipy.optimize import minimize
from astropy.convolution import Gaussian1DKernel
import scipy.integrate as integrate
from scipy.optimize import differential_evolution
import pandas as pd
import os
from scipy.integrate import simpson

In [3]:
hdul = fits.open("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/FIRST-J110949.3+124614_1_MAPPED_FLUX_SCI_LSS_U1.fits")
hdul.info()
image = hdul[0].data
image_error = hdul[1].data

Filename: C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/FIRST-J110949.3+124614_1_MAPPED_FLUX_SCI_LSS_U1.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU     363   (2094, 964)   float32   
  1  IMAGE.ERR     1 ImageHDU        44   (2094, 964)   float32   


In [4]:
image_cut = image[49:149, 0:1890]
image_error_cut = image_error[49:149, 0:1890]

In [5]:
wv = np.arange(4300.74, 4300.74+1890*1.48, 1.48)
J, N = image_cut.data.shape
#x = np.arange(0, N, 1)
y = np.arange(0, J, 1)
arcsec_y = np.arange(-58*0.256, 42*0.256, 0.256)

z = 0.04263
l_HA = 6563 * (1+z)
l_HB = 4861 * (1+z) 
l_NII_1 = 6548 * (1+z) 
l_NII_2 = 6583 * (1+z)
l_SII_1 = 6717 * (1+z)
l_SII_2 = 6731 * (1+z)
l_OIII_1 = 4959 * (1+z)
l_OIII_2 = 5007 * (1+z)

In [6]:
def calculate_bn(n):
    """ Calcola b_n per il profilo Sérsic """
    return 2*n - 1/3 + 4/(405*n) + 46/(25515*n**2)

def sersic_1d(x, params):
    """ Profilo Sérsic 1D centrato in x_0 """
    I_e, r_e, n, x_0 = params
    r = np.abs(x - x_0)
    b_n = calculate_bn(n)
    return I_e * np.exp(-b_n * ((r / r_e)**(1 / n) - 1))
    
def sersic_1d_convolved(x, params):
    """ Sérsic convoluto con PSF gaussiana """
    profile = sersic_1d(x, params)
    psf = Gaussian1DKernel(stddev = sigma, mode='center')
    conv_profile = convolve(profile, psf, mode='same')
    return conv_profile

def model_convolved(x, params1, params2, params3, params4, params5, params6):
    x_hr = np.linspace(min(x), max(x), 4000)
    profile = (
        sersic_1d(x_hr, params1) +
        sersic_1d(x_hr, params2) +
        sersic_1d(x_hr, params3) +
        sersic_1d(x_hr, params4) +
        sersic_1d(x_hr, params5) +
        sersic_1d(x_hr, params6) #+
        #sersic_1d(x_hr, params7)
    )
    kernel = Gaussian1DKernel(stddev=(sigma*len(x_hr))/len(x), x_size=len(x_hr), mode='center')   
    conv_profile = convolve(profile, kernel, mode='same')
    conv_profile_out = np.interp(x, x_hr, conv_profile)
    return conv_profile_out

In [7]:
seeing = np.loadtxt("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/PSF/Real_seeing_FIRST_1.txt")
sigma = seeing[-1] / 2.35
x_axes = np.arange(0, 100, 1)

In [8]:
def single_integral(x, params, a, b):
    # x --> array di x 
    # y --> dati sulla y (sersic_1d_conv(x))
    # params --> parametri del profilo 
    # a e b --> estremi di integrazione
    func = sersic_1d_convolved(x, params)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral 

def total_integral(x, params1, params2, params3, params4, params5, params6, a, b):
    func = model_convolved(x, params1, params2, params3, params4, params5, params6)
    mask = (x >= a)*(x <= b)
    integral = simpson(y = func[mask], x = x[mask])
    return integral  

def contamination(x, params1, params2, params3, params4, params5, params6, a, b): 
    blue_area = single_integral(x, params1, a, b) 
    green_area = single_integral(x, params2, a, b)
    magenta_area = single_integral(x, params3, a, b)
    orange_area = single_integral(x, params4, a, b)
    choco_area = single_integral(x, params5, a, b)
    purple_area = single_integral(x, params6, a, b)
    #pink_area = single_integral(x, params7, a, b)
    sum_area = blue_area + green_area + magenta_area + orange_area + choco_area + purple_area #+ pink_area
    total_area = total_integral(x, params1, params2, params3, params4, params5, params6, a, b)
    print(sum_area)
    print(total_area)
    return [(blue_area*100)/sum_area, (green_area*100)/sum_area, (magenta_area*100)/sum_area, 
            (orange_area*100)/sum_area, (choco_area*100)/sum_area, (purple_area*100)/sum_area]#, (pink_area*100)/sum_area]     

## Halpha

In [9]:
mask_HA = (wv > l_HA-8)*(wv < l_HA+8)
image_HA = image_cut[:, mask_HA]
image_error_HA = image_error_cut[:, mask_HA]
radial_profile_HA = np.sum(image_HA, axis = 1)
radial_profile_error_HA = np.sqrt(np.sum(image_error_HA**2, axis = 1))
fit_HA = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Fit/Halpha_fit.csv", index_col=0)

In [10]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_1s.csv')
df.to_csv(df_path, float_format='%.3f')

35.8410806264888
43.22323362155592
42.263846607608436
63.495859532821015
33.04101318303928
33.09375826858489
48.061741047555095
48.55272939873585
45.143032781707994
48.52529660530979
6.742976032375836
6.859373869286288
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.988429e+01   0.527940   0.138678   0.067993   0.046746   0.111642
Green      1.130263e-01  98.597244   0.302751   0.099430   0.044171   0.047088
Magenta    7.855919e-04   0.808202  72.031290  21.193454   4.234386   0.166734
Orange     7.244275e-07   0.012993  24.973102  71.973884   9.261887   0.039813
Chocolate  1.870488e-03   0.052789   2.531164   6.626785  86.258213   2.842248
Purple     2.717727e-05   0.000833   0.023015   0.038453   0.154597  96.792474


In [11]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_2s.csv')
df.to_csv(df_path, float_format='%.3f')

99.44276750799676
118.1467338555425
115.32273881223921
169.29528039202714
98.20402555893614
98.30147124809312
140.23107119489597
141.45579250735918
128.26976191623984
136.80227081236526
19.293172177585586
19.57385351956347
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.987421e+01   0.582315   0.140183   0.070000   0.049408   0.117141
Green      1.228614e-01  98.439257   0.307980   0.102806   0.046818   0.049471
Magenta    8.685122e-04   0.904206  70.850265  22.025358   4.537567   0.178281
Orange     8.161221e-07   0.014853  26.080020  70.745838  10.115919   0.043344
Chocolate  2.030951e-03   0.058447   2.598135   7.016109  85.085300   3.012197
Purple     2.952698e-05   0.000922   0.023417   0.039890   0.164988  96.599567


In [12]:
data1 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit_HA['Component 1'], fit_HA['Component 2'], fit_HA['Component 3'], fit_HA['Component 4'], fit_HA['Component 5'], fit_HA['Component 6'], fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)


data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3)
    .reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Halpha_3s.csv')
df.to_csv(df_path, float_format='%.3f')

138.93373523112197
159.95799326708593
155.04951123564294
216.3643308874278
160.53609022288398
160.539125025973
219.70996192386121
221.02948925927012
189.76927715026102
199.24928981829228
28.654278109784354
28.947611534442622
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       99.847786   0.728151   0.143491   0.074718   0.055819   0.131706
Green       0.148630  98.004145   0.320712   0.111022   0.053297   0.055812
Magenta     0.001097   1.172432  67.798510  24.092279   5.317489   0.210777
Orange      0.000001   0.020311  28.945433  67.619256  12.461279   0.053714
Chocolate   0.002450   0.073800   2.767475   8.059295  81.920966   3.479266
Purple      0.000036   0.001161   0.024379   0.043429   0.191150  96.068726


## HBeta

In [13]:
mask_HB = (wv > l_HB-8)*(wv < l_HB+8)
image_HB = image_cut[:, mask_HB] 
image_error_HB = image_error_cut[:, mask_HB]
radial_profile_HB = np.sum(image_HB, axis = 1)
radial_profile_error_HB = np.sqrt(np.sum(image_error_HB**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Fit/Hbeta_fit.csv", index_col = 0)

In [14]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_1s.csv')
df.to_csv(df_path, float_format='%.3f')

15.412809963918138
11.496312102901703
169.94700616818585
69.28463321919259
12.237830801458454
12.225922063914147
19.771580891644238
19.177187786758157
11.544695057833053
11.609767373167394
1.8170455724716859
1.8871255651278105
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.912745e+01   0.093143   0.382986   0.182996   0.224358   0.650177
Green      1.463904e-01  99.615649   0.397561   0.113413   0.077758   0.072588
Magenta    3.113610e-07   0.037262  72.098104  35.270625  14.490571   0.040133
Orange     4.685333e-01   0.208977  25.313007  61.715407  21.545830  17.798631
Chocolate  3.415463e-04   0.001020   0.650638   1.833089  61.519996   0.752030
Purple     2.572867e-01   0.043948   1.157702   0.884469   2.141486  80.686441


In [15]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_2s.csv')
df.to_csv(df_path, float_format='%.3f')

42.62102894977897
32.19480452327784
455.2212134023212
188.17228541680072
36.63405636374575
36.596740216567895
57.14338766364837
55.59304886003651
33.155134025040184
33.25567243763739
5.350276649887289
5.550240877706258
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.905191e+01   0.104532   0.384170   0.190100   0.234520   0.662723
Green      1.598103e-01  99.567147   0.401830   0.118442   0.081571   0.074116
Magenta    3.816904e-07   0.043403  71.590796  36.543911  15.344824   0.043628
Orange     5.087362e-01   0.234532  25.796533  60.245768  22.766023  18.168709
Chocolate  3.720323e-04   0.001151   0.665749   1.982975  59.333215   0.775192
Purple     2.791719e-01   0.049236   1.160921   0.918804   2.239848  80.275633


In [16]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_Hbeta_3s.csv')
df.to_csv(df_path, float_format='%.3f')


57.39574989072667
45.06056295014593
554.1906963333448
239.37116383099863
60.81677534644711
60.746347087730236
88.27269500769741
86.29938810510542
49.929759623800855
49.963225788683836
8.468403511901968
8.731698894858743
                 Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       9.882173e+01   0.143895   0.386639   0.205538   0.260005   0.698654
Green      2.010656e-01  99.398695   0.412823   0.129909   0.091312   0.078494
Magenta    6.262843e-07   0.065516  70.196403  39.196705  17.595147   0.054255
Orange     6.310671e-01   0.322827  27.131213  57.088254  26.023883  19.229655
Chocolate  4.653632e-04   0.001607   0.705530   2.386198  53.542409   0.842353
Purple     3.456726e-01   0.067460   1.167393   0.993396   2.487243  79.096590


## NII

In [17]:
mask_NII = (wv > l_NII_2-8)*(wv < l_NII_2+8)
image_NII = image_cut[:, mask_NII]
image_error_NII = image_error_cut[:, mask_NII]
radial_profile_NII = np.sum(image_NII, axis = 1)
radial_profile_error_NII = np.sqrt(np.sum(image_error_NII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Fit/NII_fit.csv", index_col = 0)

In [18]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

7.6611157248378525
8.337731511006762
40.23270445740865
57.8028212845018
15.786868022810047
15.897444012891803
27.711285516165304
27.65595391165598
17.776748410572885
19.186844987330623
2.004609971143825
1.9528145441731075
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       94.583644   0.133141   0.055368   0.021147   0.019529   0.049095
Green       0.005531  96.258181   0.016077   0.002097   0.000535   0.000084
Magenta     5.408705   3.457116  45.469904  12.545281  10.666244  32.532479
Orange      0.000747   0.143720  53.670150  85.545150  34.546517   2.035411
Chocolate   0.001294   0.007439   0.771405   1.864365  54.651606   1.309954
Purple      0.000079   0.000403   0.017097   0.021961   0.115570  64.072976


In [19]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                      fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

21.54526969773747
23.251256517801306
108.97348440051263
153.5614694943112
47.47091990460992
47.771213588224555
81.02193584949477
80.87304988413547
51.409538427027414
54.980119660931535
5.841536067862394
5.70302598660765
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       94.219011   0.148026   0.055347   0.021735   0.020288   0.050596
Green       0.006020  95.846595   0.016469   0.002196   0.000563   0.000088
Magenta     5.772686   3.834495  44.779719  12.941215  11.086906  33.515618
Orange      0.000813   0.162139  54.346791  85.028267  36.380997   2.133447
Chocolate   0.001386   0.008298   0.784502   1.983885  52.390268   1.364125
Purple      0.000085   0.000448   0.017172   0.022702   0.120978  62.936128


In [20]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_NII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

31.096349247986005
32.9819265073374
143.4880088785777
193.78765494903382
79.36747678002811
79.69579326548036
127.4192027309121
127.2078094977052
79.5432121712948
83.55199380581193
9.072556559385028
8.908679336452554
              Peak 1     Peak 2     Peak 3     Peak 4     Peak 5     Peak 6
Blue       93.314805   0.189288   0.055462   0.023141   0.021941   0.054448
Green       0.007322  94.712416   0.017626   0.002458   0.000632   0.000097
Magenta     6.675169   4.871827  42.804527  13.922370  12.007499  36.034132
Orange      0.000988   0.215196  56.282908  83.713959  40.789313   2.401790
Chocolate   0.001616   0.010698   0.822054   2.313537  47.047174   1.509814
Purple      0.000099   0.000575   0.017423   0.024535   0.133441  59.999720


## SII

In [21]:
mask_SII = (wv > l_SII_1-8)*(wv < l_SII_1+8)
image_SII = image_cut[:, mask_SII] 
image_error_SII = image_error_cut[:, mask_SII]
radial_profile_SII = np.sum(image_SII, axis = 1)
radial_profile_error_SII = np.sqrt(np.sum(image_error_SII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Fit/SII_fit.csv", index_col = 0)

In [22]:
fit

,Component 1,Component 2,Component 3,Component 4,Component 5,Component 6
I_e,0.5277,2.4995,3.7766,0.8055,2.6985,0.3420
r_e,10.0000,3.9670,5.5230,27.9262,2.3088,3.6967
n,1.7856,3.5749,0.4476,2.1522,0.7828,0.7911
x_0,16.6362,41.5833,61.3210,66.4158,72.6949,91.5461


In [23]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

6.424077821994652
6.762437556460529
36.978656622496075
64.78236911616546
13.077680385924172
13.062576392277904
23.588388509399326
24.283533589327426
13.10801903593177
13.17954963758951
1.9468548675243358
1.9553423973425768
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5  \
Blue       9.593859e+01  1.667609e-01  6.211693e-02   0.021667   0.021083   
Green      3.951279e-01  9.714518e+01  4.653224e-01   0.125301   0.096951   
Magenta    8.013831e-24  1.388983e-03  5.065671e+01  20.773284   3.516543   
Orange     3.666285e+00  2.686672e+00  4.881133e+01  78.740926  47.429845   
Chocolate  1.012210e-29  8.258669e-14  4.521359e-03   0.338819  48.935036   
Purple     7.355911e-24  8.823993e-15  1.118567e-07   0.000002   0.000542   

                 Peak 6  
Blue       3.117326e-02  
Green      1.143483e-01  
Magenta    1.261983e-08  
Orange     5.042930e+01  
Chocolate  1.683137e-05  
Purple     4.942516e+01  


In [24]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

18.276330801102873
19.127760055124256
100.47679664155186
171.24513801028667
39.282490344772455
39.23535841339363
68.94271807301982
70.89147406998406
38.36171066931178
38.54283681364065
5.7256739435836534
5.747171524299717
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5  \
Blue       9.571235e+01  1.849024e-01  6.219259e-02   0.022290   0.021655   
Green      4.187911e-01  9.684191e+01  4.683205e-01   0.129306   0.099746   
Magenta    4.342195e-23  1.984916e-03  5.018481e+01  21.220700   3.800493   
Orange     3.868859e+00  2.971205e+00  4.927876e+01  78.207766  49.102370   
Chocolate  2.043709e-29  1.461870e-13  5.917426e-03   0.419935  46.975117   
Purple     9.737146e-24  1.170030e-14  1.286363e-07   0.000003   0.000618   

                 Peak 6  
Blue       3.184678e-02  
Green      1.168509e-01  
Magenta    2.416189e-08  
Orange     5.152378e+01  
Chocolate  2.362821e-05  
Purple     4.832750e+01  


In [25]:
_HAdata1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_SII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

27.01398017403257
27.95076949383059
133.57342019476022
214.14644943111216
65.73112556698788
65.6446859179176
108.570937712404
111.44121428938612
60.812687931773034
61.01891205872345
9.11285161514374
9.13730186957342
                 Peak 1        Peak 2        Peak 3     Peak 4     Peak 5  \
Blue       9.571235e+01  2.344734e-01  6.235939e-02   0.023732   0.022888   
Green      4.187911e-01  9.602013e+01  4.762788e-01   0.138850   0.105905   
Magenta    4.342195e-23  4.186498e-03  4.890441e+01  22.163450   4.550337   
Orange     3.868859e+00  3.741209e+00  5.054669e+01  77.009741  53.092051   
Chocolate  2.043709e-29  4.440840e-13  1.026876e-02   0.664223  42.227984   
Purple     9.737146e-24  2.199462e-14  1.767016e-07   0.000004   0.000835   

                 Peak 6  
Blue       3.348434e-02  
Green      1.229508e-01  
Magenta    8.304778e-08  
Orange     5.418786e+01  
Chocolate  5.011643e-05  
Purple     4.565565e+01  


## OIII

In [26]:
mask_OIII = (wv > l_OIII_2-8)*(wv < l_OIII_2+8)
image_OIII = image_cut[:, mask_OIII] 
image_error_OIII = image_error_cut[:, mask_OIII]
radial_profile_OIII = np.sum(image_OIII, axis = 1)
radial_profile_error_OIII = np.sqrt(np.sum(image_error_OIII**2, axis = 1))
fit = pd.read_csv("C:/Users/ISAFA/Desktop/Tesi magistrale/Sources/FIRSTJ110949/FIRSTJ110949_1/Fit/OIII_fit.csv", index_col = 0)

In [27]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 1'].iloc[3] -sigma, fit_HA['Component 1'].iloc[3] + sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 2'].iloc[3] -sigma, fit_HA['Component 2'].iloc[3] + sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 3'].iloc[3] -sigma, fit_HA['Component 3'].iloc[3] + sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 4'].iloc[3] -sigma, fit_HA['Component 4'].iloc[3] + sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 5'].iloc[3] -sigma, fit_HA['Component 5'].iloc[3] + sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'], fit_HA['Component 6'].iloc[3] -sigma, fit_HA['Component 6'].iloc[3] + sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_1s.csv')
df.to_csv(df_path, float_format='%.3f')

17.75675116869985
18.462898080959125
82.87940185912355
71.45306698974838
9.894721850818247
9.885786139794329
18.432215396993435
18.33593240125839
7.748901717582729
7.9777288475176755
2.320313246784138
2.3242878679578345
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.868734e+01  1.067864e-01  1.674257e-01  6.255867e-02   0.092924   
Green      3.943713e-01  9.933456e+01  1.319039e+00  3.888097e-01   0.460540   
Magenta    1.158702e-18  1.285011e-02  7.337012e+01  2.620518e+01   6.800321   
Orange     1.232671e-10  3.313099e-05  1.048453e+01  6.066205e+01   9.391606   
Chocolate  9.182870e-01  5.457720e-01  1.465889e+01  1.268140e+01  83.254570   
Purple     1.187801e-50  3.907998e-27  5.458198e-12  3.262635e-09   0.000039   

                 Peak 6  
Blue       1.014155e-01  
Green      3.695427e-01  
Magenta    1.554938e-07  
Orange     8.243674e-04  
Chocolate  3.701779e+01  
Purple     6.251043e+01  


In [28]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -2*sigma, fit_HA['Component 1'].iloc[3] + 2*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -2*sigma, fit_HA['Component 2'].iloc[3] + 2*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -2*sigma, fit_HA['Component 3'].iloc[3] + 2*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -2*sigma, fit_HA['Component 4'].iloc[3] + 2*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                       fit_HA['Component 5'].iloc[3] -2*sigma, fit_HA['Component 5'].iloc[3] + 2*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -2*sigma, fit_HA['Component 6'].iloc[3] + 2*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_2s.csv')
df.to_csv(df_path, float_format='%.3f')

49.42799855492408
51.07406090896353
223.87807843986664
192.3161348347773
29.817080486688546
29.78788673322545
53.43250691369016
53.21250211040938
22.79496434853467
23.370850110044888
6.817373133705217
6.82769013189345
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.858326e+01  1.190060e-01  1.669557e-01  6.483466e-02   0.094881   
Green      4.266792e-01  9.925787e+01  1.320483e+00  4.039293e-01   0.470884   
Magenta    4.033937e-18  1.646387e-02  7.262040e+01  2.705178e+01   7.240392   
Orange     1.440187e-10  3.982963e-05  1.125895e+01  5.927887e+01  10.291162   
Chocolate  9.900607e-01  6.066234e-01  1.463321e+01  1.320058e+01  81.902627   
Purple     4.766343e-50  1.036986e-26  9.114280e-12  5.154768e-09   0.000054   

                 Peak 6  
Blue       1.036377e-01  
Green      3.777900e-01  
Magenta    2.604029e-07  
Orange     9.071196e-04  
Chocolate  3.785181e+01  
Purple     6.166585e+01  


In [29]:
data1 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 1'].iloc[3] -3*sigma, fit_HA['Component 1'].iloc[3] + 3*sigma)

data2 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 2'].iloc[3] -3*sigma, fit_HA['Component 2'].iloc[3] + 3*sigma)

data3 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 3'].iloc[3] -3*sigma, fit_HA['Component 3'].iloc[3] + 3*sigma)

data4 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 4'].iloc[3] -3*sigma, fit_HA['Component 4'].iloc[3] + 3*sigma)

data5 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 5'].iloc[3] -3*sigma, fit_HA['Component 5'].iloc[3] + 3*sigma)

data6 = contamination(x_axes, fit['Component 1'], fit['Component 2'], fit['Component 3'], fit['Component 4'], fit['Component 5'], fit['Component 6'],
                     fit_HA['Component 6'].iloc[3] -3*sigma, fit_HA['Component 6'].iloc[3] + 3*sigma)

data = np.concatenate([np.array(data1).reshape(-1, 1),np.array(data2).reshape(-1, 1), np.array(data3).reshape(-1, 1),
                      np.array(data4).reshape(-1, 1),np.array(data5).reshape(-1, 1), np.array(data6).reshape(-1, 1)], axis=1)

df = pd.DataFrame(data, 
                  columns=[f'Peak {i+1}' for i in range(6)],
                  index=['Blue', 'Green', 'Magenta', 'Orange', 'Chocolate', 'Purple'])
print(df)

df_path = os.path.join('Areas_OIII_3s.csv')
df.to_csv(df_path, float_format='%.3f')

68.1712300890921
69.81112339534744
280.98986732330155
241.97257224587673
50.314411251905376
50.25388496841703
82.67585883360826
82.49762384178587
36.567644433932955
37.21404104917055
10.804842355130585
10.817271005580526
                 Peak 1        Peak 2        Peak 3        Peak 4     Peak 5  \
Blue       9.828141e+01  1.595043e-01  1.656358e-01  7.010731e-02   0.098898   
Green      5.209189e-01  9.900323e+01  1.324030e+00  4.396458e-01   0.492641   
Magenta    2.785517e-17  2.992665e-02  7.060203e+01  2.893006e+01   8.374688   
Orange     2.114422e-10  6.404410e-05  1.335073e+01  5.611636e+01  12.869125   
Chocolate  1.197674e+00  8.072758e-01  1.455757e+01  1.444382e+01  78.164547   
Purple     4.036621e-49  5.686101e-26  2.310038e-11  1.237612e-08   0.000101   

                 Peak 6  
Blue       1.092288e-01  
Green      3.985965e-01  
Magenta    7.342575e-07  
Orange     1.157033e-03  
Chocolate  3.995943e+01  
Purple     5.953159e+01  
